In [29]:
# !pip install requests

In [30]:
# !pip show requests

In [31]:
# import sys
# print(sys.path)

In [32]:
# Library
import json
from dataclasses import dataclass, field
from pathlib import Path

import time
import requests
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional

In [33]:
# Parse response

@dataclass
class CodeMap:
    imo_lexical_title: str = ""
    imo_lexical_code: str = ""
    imo_confidence: str = ""
    icd10_code: str = ""
    icd10_title: str = ""
    snomed_code: str = ""
    snomed_title: str = ""


@dataclass
class NEREntity:
    id: str
    text: str
    begin: int
    end: int
    semantic: str  # problem, procedure, drug etc
    assertion: str = "present"
    section: str = ""
    origin: str = ""  
    sentence_prob: float | None = None
    concept_prob: float | None = None
    linked_entity_ids: list[str] = field(default_factory=list)
    codemap: CodeMap | None = None


@dataclass
class NERRelation:
    id: str
    semantic: str  # problem bodyloc, problem temporal etc
    from_entity_id: str
    to_entity_id: str
    from_semantic: str
    to_semantic: str

@dataclass
class ParsedNER:
    clinical_note: str
    entities: list[NEREntity]
    relations: list[NERRelation]

    def get_entity_by_span(self, begin: int, end: int) -> NEREntity | None:
        for e in self.entities:
            if e.begin == begin and e.end == end:
                return e
        return None

    def get_problems(self) -> list[NEREntity]:
        return [e for e in self.entities if e.semantic == "problem" and e.assertion == "present"]

    def get_procedures(self) -> list[NEREntity]:
        return [e for e in self.entities if e.semantic in ("procedure", "treatment") and e.assertion == "present"]

    def get_entities_with_codemaps(self) -> list[NEREntity]:
        return [e for e in self.entities if e.codemap is not None]

    def get_related_entities(self, entity: NEREntity) -> list[tuple[str, NEREntity]]:
        """Get all entities related to the given entity, with relation type."""
        related = []
        for r in self.relations:
            if r.from_entity_id == entity.id:
                target = next((e for e in self.entities if e.id == self._span_to_id(r.to_entity_id)), None)
                if target:
                    related.append((r.semantic, target))
            elif r.to_entity_id == entity.id:
                source = next((e for e in self.entities if e.id == self._span_to_id(r.from_entity_id)), None)
                if source:
                    related.append((r.semantic, source))
        return related

    @staticmethod
    def _span_to_id(span_ref: str) -> str:
        return span_ref

def parse_codemap(codemaps_raw: dict | None) -> CodeMap | None:
    if not codemaps_raw:
        return None

    imo = codemaps_raw.get("imo", {})
    icd10_codes = codemaps_raw.get("icd10cm", {}).get("codes", [])
    snomed_codes = codemaps_raw.get("snomedInternational", {}).get("codes", [])

    return CodeMap(
        imo_lexical_title=imo.get("lexical_title", ""),
        imo_lexical_code=imo.get("lexical_code", ""),
        imo_confidence=imo.get("confidence", ""),
        icd10_code=icd10_codes[0]["code"] if icd10_codes else "",
        icd10_title=icd10_codes[0]["title"] if icd10_codes else "",
        snomed_code=snomed_codes[0]["code"] if snomed_codes else "",
        snomed_title=snomed_codes[0]["title"] if snomed_codes else "",
    )

# Parse raw CCP NER JSON into structured format
def parse_ner_json(filepath: str | Path) -> ParsedNER:
    with open(filepath) as f:
        data = json.load(f)

    content = data["content"]
    entities = []
    relations = []

    for _pos, types in data["indexes"].items():
        if "Entity" in types:
            for ent_key, ent_val in types["Entity"].items():
                text = content[ent_val["begin"]:ent_val["end"]]
                attrs = ent_val.get("attrs", {})
                codemaps_raw = attrs.get("codemaps")
                codemaps = json.loads(codemaps_raw) if codemaps_raw else None

                linked = attrs.get("linked_entities", "")
                linked_ids = [lid.strip("[]") for lid in linked.split(",")] if linked else []

                entities.append(NEREntity(
                    id=ent_key,
                    text=text,
                    begin=ent_val["begin"],
                    end=ent_val["end"],
                    semantic=ent_val.get("semantic", ""),
                    assertion=attrs.get("assertion", "present"),
                    section=attrs.get("section", ""),
                    origin=attrs.get("origin", ""),
                    sentence_prob=float(attrs["sentence_prob"]) if "sentence_prob" in attrs else None,
                    concept_prob=float(attrs["concept_prob"]) if "concept_prob" in attrs else None,
                    linked_entity_ids=linked_ids,
                    codemap=parse_codemap(codemaps),
                ))

        if "Relation" in types:
            for rel_key, rel_val in types["Relation"].items():
                from_ent = rel_val["fromEnt"]
                to_ent = rel_val["toEnt"]

                # Build entity IDs from span + semantic
                from_id = f"{from_ent['begin']}_{from_ent['end']}_Entity_{from_ent['semantic']}"
                to_id = f"{to_ent['begin']}_{to_ent['end']}_Entity_{to_ent['semantic']}"

                relations.append(NERRelation(
                    id=rel_key,
                    semantic=rel_val.get("semantic", ""),
                    from_entity_id=from_id,
                    to_entity_id=to_id,
                    from_semantic=from_ent.get("semantic", ""),
                    to_semantic=to_ent.get("semantic", ""),
                ))

    return ParsedNER(clinical_note=content, entities=entities, relations=relations)


In [34]:
# test parser
parsed = parse_ner_json(r"data\ner_ext.json")
print(len(parsed.entities))

problems = parsed.get_problems()
print(f"No. of Problem entities: {len(problems)}")
for p in problems:
    imo = p.codemap.imo_lexical_title if p.codemap else "no codemap"
    print(f"{p.text} -> {imo}")

# parsed.get_procedures()
# parsed.get_related_entities()

31
No. of Problem entities: 6
ongoing issues -> Ongoing pain
somewhat low blood glucoses -> Hypoglycemia
considerable change -> Considering quitting consumption of tobacco
fatigued -> Fatigue
Diabetes, -> Diabetes mellitus
some problematic low blood glucoses -> Hypoglycemia


In [35]:
# LLM Configuration

# This is standard response from any model
@dataclass
class ModelResponse:    
    content: str
    model: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    latency_ms: float = 0.0
    raw_response: dict = field(default_factory=dict)



class OllamaLLM:
    def __init__(self, model_name: str, base_url: str = "http://localhost:11434"):
        self.model_name = model_name
        self.base_url = base_url

    def health_check(self) -> bool:
        try:
            resp = requests.get(f"{self.base_url}/api/tags", timeout=5)
            if resp.ok:
                models = [m["name"] for m in resp.json().get("models", [])]
                return any(self.model_name in m for m in models)
            return False
        except Exception:
            return False
        
    def generate(self, prompt: str, system_prompt: str = "", temperature: float = 0.1) -> ModelResponse:
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "system": system_prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "num_predict": 4096,
            },
        }

        start = time.time()
        resp = requests.post(f"{self.base_url}/api/generate", json=payload, timeout=120)
        latency = (time.time() - start) * 1000
        resp.raise_for_status()
        data = resp.json()

        return ModelResponse(
            content=data.get("response", ""),
            model=self.model_name,
            prompt_tokens=data.get("prompt_eval_count", 0),
            completion_tokens=data.get("eval_count", 0),
            latency_ms=latency,
            raw_response=data,
        )


SUPPORTED_MODELS = {
    "qwen2.5:7b": OllamaLLM,
    "llama3.1:8b": OllamaLLM,
}


def create_model(model_name: str, base_url: str = "http://localhost:11434"):    
    model_class = SUPPORTED_MODELS.get(model_name)
    if not model_class:
        raise ValueError(
            f"Unsupported model: {model_name}. "
            f"Supported: {list(SUPPORTED_MODELS.keys())}"
        )
    return model_class(model_name=model_name, base_url=base_url)


# List models available in local Ollama
def list_available_models(base_url: str = "http://localhost:11434") -> list[str]:    
    try:
        resp = requests.get(f"{base_url}/api/tags", timeout=5)
        if resp.ok:
            return [m["name"] for m in resp.json().get("models", [])]
    except Exception:
        pass
    return []

In [36]:
# Test LLM config
list_available_models()
# model_name = "qwen2.5:7b"
# ollama_url = "http://localhost:11434"

# llm = create_model(model_name)
# if not llm.health_check():
#     print(f"Model {model_name} not available. Use --model mock for testing.")
#     if print != "mock":
#         available = list_available_models(ollama_url)
#         if available:
#             print(f"Available Ollama models: {available}")
#         else:
            # print("No /llama models found. Is Ollama running?")
        

['llama3.1:8b', 'qwen2.5:7b', 'glm-5:cloud']

In [37]:
# Enrichment layer

# enriched clinical concept produced by LLM
@dataclass
class EnrichedEntity:    
    enriched_concept: str
    source_entity_ids: list[str]
    source_entity_texts: list[str]
    clinical_reasoning: str
    inference_strength: str  # explicit | strong_suggestion | weak_suggestion
    evidence_spans: list[str] = field(default_factory=list)


# full result of LLM enrichment for one clinical note
@dataclass
class EnrichmentResult:    
    model_name: str
    enriched_entities: list[EnrichedEntity]
    original_entity_count: int
    enriched_entity_count: int
    prompt_tokens: int = 0
    completion_tokens: int = 0
    latency_ms: float = 0.0
    raw_llm_response: str = ""


# Sytem Prompt

SYSTEM_PROMPT = """You are a clinical NLP expert specializing in medical coding. Your task is to analyze extracted NER entities from a clinical note and produce enriched clinical concepts by linking related entities across sentences.

RULES:
1. Only combine entities that have a genuine clinical relationship supported by the note text.
2. Produce enriched concepts that are MORE SPECIFIC than the individual entities.
3. Include laterality, severity, causation, and temporal context when present in the note.
4. Each enriched concept should cite which source entities it combines.
5. Rate inference strength:
   - "explicit": directly stated in the note (e.g., "diabetic osteomyelitis")
   - "strong_suggestion": strongly implied by clinical context (e.g., osteomyelitis + diabetes + amputation -> diabetic osteomyelitis)
   - "weak_suggestion": possible but requires clinical judgment
6. Do NOT invent information not present in the note.
7. Do NOT fix NER errors or suggest new codes — only produce enriched concept text.

Respond ONLY with valid JSON, no markdown backticks, no preamble."""


def build_entity_summary(parsed: ParsedNER) -> str:
    """Build a concise entity summary for the LLM prompt."""
    lines = []
    for ent in parsed.entities:
        if ent.codemap or ent.semantic in ("problem", "procedure", "treatment", "drug"):
            imo = ""
            icd = ""
            if ent.codemap:
                imo = f" | IMO: {ent.codemap.imo_lexical_title}"
                if ent.codemap.icd10_code:
                    icd = f" | ICD10: {ent.codemap.icd10_code} ({ent.codemap.icd10_title})"

            lines.append(
                f"  - [{ent.semantic}] \"{ent.text}\" "
                f"(pos {ent.begin}-{ent.end}, assertion={ent.assertion})"
                f"{imo}{icd}"
            )

    return "\n".join(lines)

# build relation summary for the LLM prompt
def build_relation_summary(parsed: ParsedNER) -> str:    
    lines = []
    note = parsed.clinical_note
    for rel in parsed.relations:
        # Extract text spans from entity IDs (format: begin_end_Entity_semantic)
        parts = rel.from_entity_id.split("_")
        from_text = note[int(parts[0]):int(parts[1])] if len(parts) >= 2 else ""
        parts = rel.to_entity_id.split("_")
        to_text = note[int(parts[0]):int(parts[1])] if len(parts) >= 2 else ""
        lines.append(f"  - {rel.semantic}: \"{from_text}\" -> \"{to_text}\"")
    return "\n".join(lines)

# build the full enrichment prompt from parsed NER output
def build_enrichment_prompt(parsed: ParsedNER) -> str:    
    entity_summary = build_entity_summary(parsed)
    relation_summary = build_relation_summary(parsed)

    prompt = f"""CLINICAL NOTE:
    \"\"\"
    {parsed.clinical_note}
    \"\"\"

    EXTRACTED NER ENTITIES:
    {entity_summary}

    EXTRACTED RELATIONS:
    {relation_summary}

    TASK: Analyze the clinical note and the extracted entities above. Identify entities that should be linked across sentences to form richer, more specific clinical concepts. For each enriched concept, explain the clinical reasoning.

    Respond with this exact JSON structure:
    {{
    "enriched_entities": [
        {{
        "enriched_concept": "The enriched clinical concept text (e.g., 'Osteomyelitis of left great toe due to poorly controlled type 2 diabetes mellitus')",
        "source_entities": ["entity text 1", "entity text 2"],
        "clinical_reasoning": "Why these entities are clinically related",
        "inference_strength": "explicit | strong_suggestion | weak_suggestion",
        "evidence_spans": ["exact text from the note supporting this link"]
        }}
    ]
    }}"""

    return prompt


# Enrichment Execution
# Parse LLM JSON response and handling issues
def parse_llm_response(response_text: str) -> list[dict]:
    text = response_text.strip()

    # Strip markdown code fences if present
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text[3:]
    if text.endswith("```"):
        text = text[:-3]
    if text.startswith("json"):
        text = text[4:]
    text = text.strip()

    try:
        data = json.loads(text)
        return data.get("enriched_entities", [])
    except json.JSONDecodeError as e:
        print(f"Failed to parse LLM response as JSON: {e}")
        print(f"Raw response: {text[:500]}")
        return []

# Match LLM reported source entity texts back to NER entity IDs
def match_source_entities(source_texts: list[str], parsed: ParsedNER) -> list[str]:
    matched_ids = []
    for src in source_texts:
        src_lower = src.lower().strip()
        for ent in parsed.entities:
            if ent.text.lower().strip() == src_lower or src_lower in ent.text.lower():
                matched_ids.append(ent.id)
                break
    return matched_ids

# Main enrichment function.
def enrich_entities(parsed: ParsedNER, llm, temperature = 0.1) -> EnrichmentResult:
    prompt = build_enrichment_prompt(parsed)
    print(f"Sending enrichment prompt to {llm.model_name} ({len(prompt)} chars)")

    response: ModelResponse = llm.generate(
        prompt=prompt,
        system_prompt=SYSTEM_PROMPT,
        temperature=temperature,
    )

    print(f"LLM response: {response.completion_tokens} tokens, {response.latency_ms:.0f}ms")

    raw_entities = parse_llm_response(response.content)

    enriched = []
    for raw in raw_entities:
        source_texts = raw.get("source_entities", [])
        source_ids = match_source_entities(source_texts, parsed)

        enriched.append(EnrichedEntity(
            enriched_concept=raw.get("enriched_concept", ""),
            source_entity_ids=source_ids,
            source_entity_texts=source_texts,
            clinical_reasoning=raw.get("clinical_reasoning", ""),
            inference_strength=raw.get("inference_strength", "weak_suggestion"),
            evidence_spans=raw.get("evidence_spans", []),
        ))

    codemap_entities = parsed.get_entities_with_codemaps()

    return EnrichmentResult(
        model_name=llm.model_name,
        enriched_entities=enriched,
        original_entity_count=len(codemap_entities),
        enriched_entity_count=len(enriched),
        prompt_tokens=response.prompt_tokens,
        completion_tokens=response.completion_tokens,
        latency_ms=response.latency_ms,
        raw_llm_response=response.content,
    )


In [38]:
# Output Generation
def generate_enriched_json(parsed: ParsedNER, result: EnrichmentResult, output_path: str | Path) -> dict:
    output = {
        "metadata": {
            "model": result.model_name,
            "original_entity_count": result.original_entity_count,
            "enriched_entity_count": result.enriched_entity_count,
            "prompt_tokens": result.prompt_tokens,
            "completion_tokens": result.completion_tokens,
            "latency_ms": round(result.latency_ms, 1),
        },
        "enriched_entities": [],
    }

    for ee in result.enriched_entities:
        # Gather source entity details
        source_details = []
        for eid in ee.source_entity_ids:
            ent = next((e for e in parsed.entities if e.id == eid), None)
            if ent:
                detail = {
                    "id": ent.id,
                    "text": ent.text,
                    "semantic": ent.semantic,
                    "assertion": ent.assertion,
                }
                if ent.codemap:
                    detail["original_imo"] = {
                        "lexical_title": ent.codemap.imo_lexical_title,
                        "lexical_code": ent.codemap.imo_lexical_code,
                    }
                    if ent.codemap.icd10_code:
                        detail["original_icd10"] = {
                            "code": ent.codemap.icd10_code,
                            "title": ent.codemap.icd10_title,
                        }
                source_details.append(detail)

        output["enriched_entities"].append({
            "enriched_concept": ee.enriched_concept,
            "inference_strength": ee.inference_strength,
            "clinical_reasoning": ee.clinical_reasoning,
            "evidence_spans": ee.evidence_spans,
            "source_entities": source_details,
            "source_entity_texts": ee.source_entity_texts,
        })

    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)

    return output


In [39]:
# Execute enrichment

model_name = "qwen2.5:7b"
ollama_url = "http://localhost:11434"

llm = create_model(model_name)
result = enrich_entities(parsed, llm)

Sending enrichment prompt to qwen2.5:7b (5230 chars)
LLM response: 153 tokens, 39024ms


In [40]:
# Extract output
safe_model = model_name.replace(":", "_").replace("/", "_")

# Enriched JSON
json_path = f"enriched_{safe_model}.json"
enriched_data = generate_enriched_json(parsed, result, json_path)
print(f"Enriched JSON -> {json_path}")

Enriched JSON -> enriched_qwen2.5_7b.json
